# Simulador RRWPT — área de protección de pozos (un pozo, 3D 450×450×4)

PANEL de variables **idéntico a `Main_rrwpt.ipynb`** (más las variables de
**conditioning**), con la configuración del ejemplo: **un solo pozo** y malla
**450 × 450 × 4**.

Flujo: **campo verdadero** (sección 2) → **campos condicionados** (sección 3) →
**simulación de un escenario** con el campo condicionado de la iteración 1
(sección 4), que produce el campo probabilístico de protección. Durante la
simulación se imprime el avance **paso a paso (QS)** para ver en qué QS va.

> ⚠️ A 450×450×4 cada realización tarda varios minutos. Baja `n_pts_x/y` o
> `trns.npartic_QS` en el PANEL si quieres iterar rápido.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from rrwpt import config, pipeline

# =============================================================================
# PARÁMETROS DEL MODELO — valores originales de main/MC_config_RRWPT.m
# (se omiten optimización PSO/OMOPSO, GSA/Sobol, PCE, FPCA, LHS y ríos
#  transitorios, no utilizados de momento)
# =============================================================================

params = {

    # ---- Discretización del dominio (MC_config líneas 82-97) ---------------
    "Two_D":      0,        # 1 => 2D, 0 => 3D (MC_config_RRWPT.m usa 0: los Qp están calibrados para el espesor 3D de 60 m; en 2D el área de captura se infla ~60x)
    "n_pts_x":    450,      # número de elementos en x
    "n_pts_y":    450,      # número de elementos en y
    "n_pts_z":    4,        # número de elementos en z (solo 3D)
    "d_pts_x":    15.0,     # [m] tamaño de celda en x (debe ser cuadrada)
    "d_pts_y":    15.0,     # [m] tamaño de celda en y
    "d_pts_z":    15.0,     # [m] tamaño de celda en z
    "space_disc": 2,        # pasos de espacio por celda (dX = d_pts/space_disc)

    # ---- Pozos de bombeo (líneas 148-169) -----------------------------------
    "inP": np.array([0.80, 0.50, 0.70]),      # [x% y% z%] pozo principal
    "Ex_pump": 0,                             # 0 => UN SOLO POZO (solo inP)
    "Ex_inP": np.array([[0.80, 0.45],         # [x% y%] pozos extra (máx. 10)
                        [0.80, 0.55],
                        [0.80, 0.60],
                        [0.80, 0.40],
                        [0.80, 0.35]]),
    "well_radius":     15,    # [m] radio físico del pozo
    "use.circular":    1,     # inyección circular (1) / rectangular (0)
    "use.longinject":  4,     # [celdas] radio de inyección de partículas

    # ---- Discretización temporal (líneas 183-196) ---------------------------
    "n_reali":          500,    # realizaciones espaciales (Monte Carlo)
    "t_crit":           1,      # activar tiempo crítico (borrar partículas)
    "crit.t_crit":      180,    # [d] tiempo de viaje de la zona de protección (t50)
    "tim.tend":         360,    # [d] duración total de la simulación
    "trns.npartic_QS":  1000,   # partículas por pozo por paso QS
    "dti":              1800,   # [s] paso de tiempo de transporte (NO CAMBIAR)
    "tim.dtvis":        86400,  # [s] paso de visualización (NO CAMBIAR)
    "tim.deltQS":       10,     # [d] longitud del paso cuasi-estacionario
    "TTI":              1,      # [d] intervalo de inyección de partículas

    # ---- Transporte: dispersión/difusión (líneas 209-217) -------------------
    "RWPTtrans":   1,       # transporte por RWPT
    "dispersion":  1,       # transporte tipo Scheidegger (con dispersión)
    "source_geo":  1,       # 1 fuente puntual (2 rectangular y 3 gaussiana: no migradas)
    "trns.n":      0.35,    # [-] porosidad
    "trns.at":     0.10,    # [m] dispersividad transversal (>= media celda)
    "trns.al":     1.00,    # [m] dispersividad longitudinal (~10*at)
    "trns.Dm":     1e-9,    # [m²/s] coeficiente de difusión molecular
    "R4":          0,       # 0 Euler (RK4 no migrado)

    # ---- Parámetros espaciales: geoestadística (líneas 219-232, 392-408) ----
    "het":      1,          # 1 heterogéneo (geoestadístico), 0 homogéneo
    "het_geo":  0,          # enfoque geológico (no migrado)
    "use_variable_logK":          0,   # media log-K aleatoria por realización
    "use_variable_logK_variance": 0,   # varianza aleatoria por realización
    "stru.muA":   -5.5,     # frontera inferior de la media de log-K
    "stru.muE":   -7.5,     # frontera superior de la media de log-K
    "stru.variA":  0.50,    # frontera inferior de la varianza
    "stru.variE":  0.70,    # frontera superior de la varianza
    "use_var_matern_kappa": 1,   # variograma Matérn (0 => gaussiano)
    "use_var_corr_length":  0,   # longitud de correlación aleatoria
    "use_var_shapefactor":  0,   # kappa aleatorio
    "stru.intexA": 450.0,   # [m] escala integral en x (frontera inferior)
    "stru.intexE": 390.0,   # [m] escala integral en x (frontera superior)
    "stru.inteyA": 110.0,   # [m] escala integral en y (frontera inferior)
    "stru.inteyE": 160.0,   # [m] escala integral en y (frontera superior)
    "stru.intezA":  25.0,   # [m] escala integral en z (solo 3D)
    "stru.intezE":  25.0,
    "stru.kapA":    0.4,    # kappa Matérn (frontera inferior; 0.5 = exponencial)
    "stru.kapE":    0.6,    # kappa Matérn (frontera superior)

    # ---- Escenario de referencia / superposición (líneas 236-251) -----------
    "par0.dh":    0.01,     # [-] gradiente de referencia (DH por 100 m)
    "par0.Qp":    0.001,    # [m³/s] bombeo de referencia
    "par0.qr":    10,       # [mm/a] recarga de referencia
    "par0.dirh":  0.0,      # [°] dirección de referencia
    "tim.superpos":     1,  # usar superposición (FEM_CODE=0)
    "tim.sup_dirh":     1,  # superponer dirección del flujo
    "tim.sup_dh":       1,  # superponer gradiente
    "tim.sup_qpump":    1,  # superponer bombeo
    "tim.sup_recharge": 1,  # superponer recarga
    "FEM_CODE":         0,  # 0 superposición (1 FEM directo por QS: no migrado)

    # ---- Drivers transitorios sinusoidales (líneas 264-366) -----------------
    "use_knownTprog": 0,    # 0 => comportamiento sinusoidal, 1 => constante
    "synth":          1,    # 1 => valores promedio de los rangos (deterministas)
    # rangos de la sinusoide
    "tim.T_amp_a":     0,      # [%] amplitud mínima (0 = estado estacionario)
    "tim.T_amp_e":     100,    # [%] amplitud máxima (100 = amplitud completa)
    "tim.T_fr_a":      1.00,   # [-] frecuencia de oscilación (ciclos por t_total)
    "tim.T_fr_e":      1.00,
    "tim.T_phase_a":   0,      # [°] fase inicial
    "tim.T_phase_e":   360,
    # 3.1 flujo de fondo
    "tim.dhA":    0.0015,   # [-] gradiente hidráulico mínimo
    "tim.dhE":    0.0065,   # [-] gradiente hidráulico máximo
    "tim.dirhA":  160,      # [°] dirección mínima del flujo de fondo
    "tim.dirhE":  200,      # [°] dirección máxima
    # 3.3 bombeo
    "tim.QpA":    0.005,    # [m³/s] bombeo mínimo
    "tim.QpE":    0.05,     # [m³/s] bombeo máximo
    # 3.4 recarga natural
    "tim.qr0A":   50,       # [mm/a] recarga mínima
    "tim.qr0E":   500,      # [mm/a] recarga máxima

    # ---- Conditioning: campos K condicionados a muestras de log-K -----------
    "cond.Kflag":      1,       # 1 => condicionar a muestras (kriging), 0 => no
    "cond.n_cond":     10,      # nº de ubicaciones de muestreo (en 3D toma toda la columna)
    "cond.r_K":        1.0,     # varianza del error de medición
    "cond.seed_field": None,    # semilla del campo verdadero (None = aleatorio cada corrida)
    "cond.seed_obs":   None,    # semilla de las ubicaciones  (None = aleatorio cada corrida)
}

ctrl = config.make_ctrl(**params)
rng = np.random.default_rng()   # MATLAB: RandStream('mt19937ar', seed=clock)

config.finalize_ctrl(ctrl)
print(f"Malla {ctrl.n_pts_y}x{ctrl.n_pts_x} | QS steps: {ctrl.tim.tstep} | "
      f"particulas/QS/pozo: {ctrl.trns.npartic_QS} | "
      f"pozos: {1 + (len(ctrl.Ex_inP) if ctrl.Ex_pump else 0)}")

# ---- Isolíneas de la WHPA probabilística (SOLO en el Monte Carlo) -------
# WHPA al nivel de protección L% = región con probabilidad >= (100-L)%
# (Rodríguez-Pretelín & Nowak 2018): a mayor nivel de protección, MAYOR área.
#   90 -> mayor área (prob>=10) ; 50 -> prob>=50 ; 10 -> menor área (prob>=90)
ISOLINEAS_MC = [10, 50, 90]


## 1. Inicialización (malla, FEM, BCs, modelo, muestras de conditioning)

`pipeline.setup` construye la malla y, si el conditioning está activo, el **campo
verdadero** y sus muestras.

In [ ]:
import time
from rrwpt import geostat

grid, flowpar, model, ctrl = pipeline.setup(ctrl, rng)

# utilidades de graficado
extent = [0, grid.n_pts[1]*grid.d_pts[1], grid.n_pts[0]*grid.d_pts[0], 0]   # [m]
n1   = (grid.n_pts[0]+1, grid.n_pts[1]+1)
to2d = lambda v: np.asarray(v).reshape(n1, order="F")
def capa_sup(v):
    """Capa superficial (z=0) de un campo por elemento; en 2D devuelve el plano."""
    F = np.asarray(v).reshape(tuple(grid.n_pts), order="F")
    return F[:, :, 0] if grid.nd == 3 else F
wells_frac = [ctrl.inP[:2]] + (list(ctrl.Ex_inP) if ctrl.Ex_pump else [])
wells_xy = [(round(grid.data.n_el[1]*f[0])*grid.d_pts[1],
             round(grid.data.n_el[0]*f[1])*grid.d_pts[0]) for f in wells_frac]
cell_km2 = grid.d_pts[0]*grid.d_pts[1] / 1e6

def nuevo_rng():
    """rng aleatorio por realización (None de semilla => distinto cada corrida, como MATLAB)."""
    return np.random.default_rng()

if ctrl.cond.Kflag == 1:
    li = ctrl.cond.loc_idx
    samp_x = grid.x_pts[1].reshape(-1, order="F")[li]
    samp_y = grid.x_pts[0].reshape(-1, order="F")[li]
    print(f"Conditioning: {ctrl.cond.loc_idx.size} ubicaciones × {ctrl.cond.n_layers} capa(s) "
          f"= {ctrl.cond.obs_idx.size} muestras | r_K={ctrl.cond.r_K} | "
          f"rango medido [{ctrl.cond.yobs.min():.2f}, {ctrl.cond.yobs.max():.2f}]")
else:
    samp_x = samp_y = None
    print("Conditioning desactivado (cond.Kflag=0).")

## 2. Campo verdadero (de referencia)

El **campo "verdadero"** sintético del que se **leen las muestras** de log-K en las
`cond.n_cond` ubicaciones (puntos rojos). En un estudio real no se conoce; aquí se
muestra para entender el experimento. Capa superficial (z = 0).

In [ ]:
if ctrl.cond.Kflag == 1:
    fig, ax = plt.subplots(figsize=(6.8, 5.6))
    im = ax.imshow(capa_sup(ctrl.cond.Y_ref), extent=extent, aspect="equal", cmap="viridis")
    for wx, wy in wells_xy:
        ax.plot(wx, wy, "w^", ms=10, mec="black")
    ax.scatter(samp_x, samp_y, c="red", s=55, marker="o", edgecolors="white", linewidths=1.0,
               label="muestras")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_title("Campo VERDADERO de referencia — capa z=0\n(solo origen de las muestras; no se usa en la simulación)")
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    plt.colorbar(im, ax=ax, shrink=0.85, label="log-K (m/s)")
    plt.show()
else:
    print("Sin campo verdadero (conditioning desactivado).")

## 3. Campos condicionados (ejemplos)

`N_CAMPOS` realizaciones **condicionadas** al campo verdadero de la sección 2:
difieren entre sí lejos de los datos, pero **todas pasan por las mismas ubicaciones
de muestreo** (rojo). Capa superficial (z = 0).

In [ ]:
N_CAMPOS = 4    # nº de campos condicionados a generar/mostrar

campos = [geostat.uncsim_kfield(ctrl, model, grid, rng=nuevo_rng()) for _ in range(N_CAMPOS)]
vmin = min(c.min() for c in campos); vmax = max(c.max() for c in campos)

fig, axes = plt.subplots(1, N_CAMPOS, figsize=(4.0*N_CAMPOS, 4.2)); axes = np.atleast_1d(axes)
for k, (ax, c) in enumerate(zip(axes, campos)):
    im = ax.imshow(capa_sup(c), extent=extent, aspect="equal", cmap="viridis", vmin=vmin, vmax=vmax)
    for wx, wy in wells_xy:
        ax.plot(wx, wy, "w^", ms=7, mec="black")
    if samp_x is not None:
        ax.scatter(samp_x, samp_y, c="red", s=28, marker="o", edgecolors="white", linewidths=0.7)
    ax.set_title(f"condicionado {k+1}"); ax.set_xlabel("x [m]")
axes[0].set_ylabel("y [m]")
fig.colorbar(im, ax=axes.tolist(), shrink=0.75, label="log-K (m/s)")
fig.suptitle(f"{N_CAMPOS} campos condicionados (capa z=0) — todos honran las muestras", y=1.03)
plt.show()

if ctrl.cond.Kflag == 1:
    errs = np.array([np.abs(c[ctrl.cond.obs_idx] - ctrl.cond.yobs).mean() for c in campos])
    print(f"error medio en las muestras por campo: {np.round(errs, 2)}  (≈ raíz(r_K)={np.sqrt(ctrl.cond.r_K):.2f})")

## 4. Simulación de UN escenario (demostración del simulador)

Se simula **un solo escenario** usando el **campo condicionado de la iteración 1**
(el "condicionado 1" de la sección 3). Durante la corrida se imprime el **avance
paso a paso (QS)** — así sabes en qué QS va.

Resultado: el **campo probabilístico de protección** (`A50temporal`, % de pasos QS)
**sobre el campo de conductividad usado** (fondo) y con las **muestras** marcadas.
Además se visualiza la **zona de captura en cada paso QS**.

In [ ]:
# campo condicionado de la ITERACIÓN 1 (el primero de la sección 3)
yk1 = campos[0]

print("Simulando UN escenario con el campo condicionado de la iteración 1...")
print("(verás el avance por paso QS abajo)\n")
t0 = time.time()
res1 = pipeline.run_realization(ctrl, grid, model, flowpar, rng=nuevo_rng(),
                                verbose=True, yk=yk1)
print(f"\nEscenario simulado en {(time.time()-t0)/60:.1f} min")
A50_1 = res1.A50temporal     # campo probabilístico de protección (% de pasos QS)

In [ ]:
# --- RESULTADO: área de protección sobre el campo de conductividad usado ---
fig, ax = plt.subplots(figsize=(7.8, 6.4))
# fondo: campo de conductividad condicionado (capa superficial)
imk = ax.imshow(capa_sup(yk1), extent=extent, aspect="equal", cmap="gray")
# encima: área de protección probabilística (transparente donde es 0)
A = to2d(A50_1); A = np.ma.masked_where(A <= 0, A)
imp = ax.imshow(A, extent=extent, aspect="equal", cmap="hot_r", vmin=0, vmax=100, alpha=0.80)
# muestras de conductividad + pozo
ax.scatter(samp_x, samp_y, c="cyan", s=60, marker="o", edgecolors="black", linewidths=1.0,
           label="muestras de K")
for k, (wx, wy) in enumerate(wells_xy):
    ax.plot(wx, wy, "b^", ms=13, mec="white", label="pozo" if k == 0 else None)
ax.legend(loc="upper right", fontsize=8)
ax.set_title("Área de protección probabilística (escenario 1)\n"
             "fondo = campo de conductividad condicionado usado")
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
plt.colorbar(imk, ax=ax, shrink=0.85, label="log-K (m/s)  [fondo]", location="left", pad=0.12)
plt.colorbar(imp, ax=ax, shrink=0.85, label="% de pasos QS (protección)")
plt.show()

### Zona de captura en cada paso QS

Cada panel es la zona de captura del pozo en ese paso cuasi-estacionario (solo
existen a partir de `t_crit`). Permite ver cómo se construye el área de protección
paso a paso.

In [ ]:
qs = res1.qs_maps[0]                # pozo principal (único)
nq = len(qs)
ncol = 6; nrow = int(np.ceil(nq/ncol))
t0qs = int(ctrl.crit.t_crit/ctrl.tim.deltQS)   # nº de QS antes del primer mapa emitido
fig, axes = plt.subplots(nrow, ncol, figsize=(2.4*ncol, 2.5*nrow))
axes = np.atleast_1d(axes).ravel()
for k in range(nq):
    axes[k].imshow(to2d(qs[k]), extent=extent, aspect="equal", cmap="Greys")
    for wx, wy in wells_xy:
        axes[k].plot(wx, wy, "r^", ms=5)
    axes[k].set_title(f"QS {t0qs+k+1}/{ctrl.tim.tstep}", fontsize=8)
    axes[k].set_xticks([]); axes[k].set_yticks([])
for k in range(nq, len(axes)):
    axes[k].axis("off")
fig.suptitle("Zona de captura por paso cuasi-estacionario (QS) — escenario 1", y=1.01)
plt.tight_layout(); plt.show()

### Video de las partículas (escenario de ejemplo)

Se re‑simula el mismo escenario **grabando las posiciones de TODAS las partículas**
en cada **intervalo de inyección (TTI)**. El video (GIF) las muestra desde su
**aparición en el pozo** hasta su **eliminación** (al acumular `t_crit` días),
**coloreadas por edad**, sobre el **campo de conductividad** (fondo). Así se observa
el efecto de la conductividad y de los **drivers transitorios** (la dirección del
flujo, en el título, cambia entre pasos QS) sobre su desplazamiento.

In [ ]:
import os
import imageio.v2 as imageio

# re-simular el escenario de ejemplo (mismo campo condicionado yk1) GRABANDO partículas
print("Re-simulando el escenario de ejemplo y grabando partículas para el video...")
res_vid = pipeline.run_realization(ctrl, grid, model, flowpar, rng=nuevo_rng(),
                                   verbose=False, yk=yk1, record_particles=True)
frames = res_vid.particle_frames[0]            # pozo único
print(f"{len(frames)} frames grabados (uno por intervalo de inyección TTI = {ctrl.TTI} d)")

VID_DIR = os.path.join("resultados", "video_particulas"); os.makedirs(VID_DIR, exist_ok=True)
gif_path = os.path.join(VID_DIR, "particulas_ejemplo.gif")
Kbg = capa_sup(yk1)                             # fondo: campo de conductividad usado
tcd = ctrl.crit.t_crit                          # edad máxima (días) para el color

import time; t0 = time.time()
imgs = []
for i, f in enumerate(frames):
    fig, ax = plt.subplots(figsize=(6.0, 5.0))
    ax.imshow(Kbg, extent=extent, aspect="equal", cmap="gray")
    X = f["X"]; ageD = f["age"]/86400.0; al = f["alive"]
    sc = ax.scatter(X[al, 1], X[al, 0], c=ageD[al], cmap="turbo", vmin=0, vmax=tcd,
                    s=0.6, alpha=0.6, linewidths=0)
    for wx, wy in wells_xy:
        ax.plot(wx, wy, "r^", ms=10, mec="white")
    cb = fig.colorbar(sc, ax=ax, shrink=0.85); cb.set_label("edad [días]")
    ax.set_xlim(0, extent[1]); ax.set_ylim(extent[2], 0); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"día {(i+1)*ctrl.TTI} | {int(al.sum())} partículas | flujo {f['HeadDir']:.0f}°", fontsize=9)
    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    imgs.append(np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)[:, :, :3].copy())
    plt.close(fig)

imageio.mimsave(gif_path, imgs, fps=15, loop=0)
print(f"Video: {len(imgs)} frames | {os.path.getsize(gif_path)/1e6:.1f} MB | "
      f"render {time.time()-t0:.0f}s | guardado en {gif_path}")

from IPython.display import Image
Image(gif_path)

## 5. Construcción de la simulación por Monte Carlo (condicionado, reanudable)

El ensamble genera, en cada iteración, un **campo de conductividad condicionado** al
**mismo campo verdadero** (mismas muestras) y simula su transporte. El mapa de
**protección probabilística** se va promediando sobre las realizaciones.

- **Reanudable:** cada realización se guarda en `resultados/montecarlo_condicionado/`.
  Si interrumpes la celda, al re‑ejecutarla **continúa desde la última** realización.
- **Campo verdadero fijo:** el ensamble guarda/recarga el campo verdadero y sus
  muestras, para que TODAS las realizaciones (incluso tras reanudar) se condicionen
  al mismo campo. Para empezar un ensamble nuevo, **borra esa carpeta**.
- **Isolíneas WHPA** (`ISOLINEAS_MC` en el PANEL): la WHPA al **nivel de protección
  L %** es la región con probabilidad **≥ (100−L) %** (Rodríguez‑Pretelín & Nowak
  2018). Por eso, a **mayor** nivel de protección, **mayor** área: la isolínea
  **90 %** encierra la mayor área (prob ≥ 10 %), la **50 %** es prob ≥ 50 %, y la
  **10 %** es el núcleo (prob ≥ 90 %).

In [ ]:
import os
from IPython.display import clear_output

N_MC   = 10     # nº de realizaciones del ensamble (súbelo para más resolución; se reanuda)
MC_DIR = os.path.join("resultados", "montecarlo_condicionado")
os.makedirs(MC_DIR, exist_ok=True)

# fijar el campo verdadero del ensamble: se guarda la 1ª vez y se recarga al reanudar
_cp = os.path.join(MC_DIR, "conditioning.npz")
if os.path.exists(_cp):
    d = np.load(_cp)
    ctrl.cond.Y_ref = d["Y_ref"]; ctrl.cond.obs_idx = d["obs_idx"]; ctrl.cond.yobs = d["yobs"]
    ctrl.cond.loc_idx = d["loc_idx"]; ctrl.cond.n_layers = int(d["n_layers"])
    ctrl.cond.cov_cache = None; ctrl.cond.fftqe = None
    print(f"Reanudando ensamble: campo verdadero y muestras cargados de {_cp}")
else:
    np.savez_compressed(_cp, Y_ref=ctrl.cond.Y_ref, obs_idx=ctrl.cond.obs_idx,
                        yobs=ctrl.cond.yobs, loc_idx=ctrl.cond.loc_idx, n_layers=ctrl.cond.n_layers)
    print(f"Nuevo ensamble: campo verdadero y muestras guardados en {_cp}")
print(f"Carpeta del ensamble: {MC_DIR}  (bórrala para empezar un ensamble nuevo)")

In [ ]:
def _rp(l):
    return os.path.join(MC_DIR, f"reali_{l:03d}.npz")

def _plot_whpa(P, titulo):
    """Mapa de protección probabilístico + isolíneas WHPA.
    Nivel de protección L% = isolínea en el contorno de probabilidad (100-L)%."""
    P2 = to2d(P)
    fig, ax = plt.subplots(figsize=(7.2, 5.9))
    im = ax.imshow(P2, extent=extent, aspect="equal", cmap="hot_r", vmin=0, vmax=100)
    if ISOLINEAS_MC:
        niveles_prob = sorted(set(int(100 - L) for L in ISOLINEAS_MC))   # contornos en probabilidad
        paleta = {int(100-L): c for L, c in zip(sorted(ISOLINEAS_MC),
                  ["#000000", "#2ca02c", "#1f77b4", "#9467bd", "#ff7f0e"])}
        cs = ax.contour(P2, levels=niveles_prob, colors=[paleta[v] for v in niveles_prob],
                        extent=[extent[0], extent[1], extent[3], extent[2]],
                        origin="upper", linewidths=1.7)
        ax.clabel(cs, fmt={int(100-L): f"{L}%" for L in ISOLINEAS_MC}, fontsize=8)
    for wx, wy in wells_xy:
        ax.plot(wx, wy, "b^", ms=10, mec="white")
    if samp_x is not None:
        ax.scatter(samp_x, samp_y, c="cyan", s=22, marker="o", edgecolors="black", linewidths=0.5)
    ax.set_title(titulo); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    plt.colorbar(im, ax=ax, shrink=0.85, label="probabilidad [%]")
    plt.show()

# reanudación: cargar lo ya simulado
A50_acc = [np.load(_rp(l))["A50"].astype(float) for l in range(N_MC) if os.path.exists(_rp(l))]
print(f"Realizaciones ya en disco: {len(A50_acc)}/{N_MC}")

t0 = time.time()
for l in range(N_MC):
    if os.path.exists(_rp(l)):
        continue
    yk_l = geostat.uncsim_kfield(ctrl, model, grid, rng=nuevo_rng())   # campo condicionado de la iteración l
    r = pipeline.run_realization(ctrl, grid, model, flowpar, rng=nuevo_rng(), yk=yk_l, verbose=False)
    tmp = _rp(l) + ".tmp.npz"                                          # guardado atómico (reanudable)
    np.savez_compressed(tmp, A50=r.A50temporal.astype(np.float32)); os.replace(tmp, _rp(l))
    A50_acc.append(r.A50temporal)
    # VISUALIZADOR EN VIVO (interactivo): mapa probabilístico acumulado con isolíneas
    clear_output(wait=True)
    _plot_whpa(np.mean(A50_acc, axis=0),
               f"Monte Carlo condicionado — {len(A50_acc)}/{N_MC} realizaciones  "
               f"({(time.time()-t0)/60:.1f} min)")
print(f"Ensamble: {len(A50_acc)}/{N_MC} realizaciones | nuevo cómputo {(time.time()-t0)/60:.1f} min")

### Resultado del ensamble — WHPA probabilística con isolíneas

Mapa final de protección probabilística del ensamble, con las **isolíneas de la
WHPA** (`ISOLINEAS_MC`) y el área encerrada por cada **nivel de protección**.

In [ ]:
A50_mc = [np.load(_rp(l))["A50"].astype(float) for l in range(N_MC) if os.path.exists(_rp(l))]
P_conjunta = np.mean(A50_mc, axis=0)
_plot_whpa(P_conjunta, f"Área de protección probabilística — ensamble de {len(A50_mc)} realizaciones")

print(f"{'WHPA':>7} | {'área [km²]':>11} | criterio")
print("-"*56)
print(f"{'>0%':>7} | {(P_conjunta>0).sum()*cell_km2:>11.3f} | envolvente (prob > 0)")
for L in sorted(ISOLINEAS_MC):
    thr = 100 - L
    print(f"{L:>6}% | {(P_conjunta>=thr).sum()*cell_km2:>11.3f} | nivel de protección {L}% (prob >= {thr}%)")

## 6. Resumen

- **Campo verdadero** → se muestrea en `cond.n_cond` ubicaciones.
- **Campos condicionados** → honran esas muestras (hipótesis del acuífero).
- **Escenario de demostración** (sección 4) → un campo condicionado → su área de
  protección sobre la conductividad usada, con el avance por QS.
- **Monte Carlo condicionado** (sección 5) → ensamble de realizaciones (reanudable,
  guardado en disco) → **área de protección probabilística** del acuífero
  restringida por los datos.